## Codificar programas faltantes de las bases  de censos universitarios y TUNI

In [2]:
# Instalar librerías
!pip install -q pandas numpy scikit-learn sentence-transformers torch

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
httpcore 1.0.5 requires h11<0.15,>=0.13, but you have h11 0.16.0 which is incompatible.

[notice] A new release of pip available: 22.3.1 -> 26.3
[notice] To update, run: C:\Users\Usuario\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [4]:
# Paquetes
from pathlib import Path
import pandas as pd
import numpy as np
import re

from sklearn.neighbors import NearestNeighbors
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

In [5]:
# Cargar entrenamiento
carpeta = Path(
    r"D:\Daniel_Pajita\1_Proyectos\Research\Educacion"
    r"\educacion-universitaria-peru\2_Datos"
    r"\Correspondencias_entre_clasificaciones"
)

df_train = pd.read_csv(carpeta / "BD_tuni_programas_entrenamiento.txt",sep="|",dtype=str,encoding="utf-8-sig",)

# Combinar los programas pendientes de clasificación
archivos_prediccion = [
    "BD_tuni_programas_predicción.txt",
    "programas_censo1996.txt",
    "programas_censo2010.txt",
]

df_pred = pd.concat(
    [
        pd.read_csv(carpeta / archivo,sep="|",dtype=str,encoding="utf-8-sig",)
        for archivo in archivos_prediccion
    ],
    ignore_index=True,
)

df_pred = (df_pred.drop_duplicates(subset=["nivel_academico", "nombre_programa"]).reset_index(drop=True))

print(f"Entrenamiento: {len(df_train):,} programas")
print(f"Predicción: {len(df_pred):,} programas")

Entrenamiento: 5,122 programas
Predicción: 2,330 programas


In [6]:
# Limpieza 
def clean_text(s):
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

for df in [df_train, df_pred]:
    df["nivel_academico_clean"] = df["nivel_academico"].apply(clean_text)
    df["nombre_programa_clean"] = df["nombre_programa"].apply(clean_text)

# Texto final que usará el modelo (concatena nivel + nombre)
df_train["texto_modelo"] = df_train["nivel_academico_clean"] + " | " + df_train["nombre_programa_clean"]
df_pred["texto_modelo"]  = df_pred["nivel_academico_clean"]  + " | " + df_pred["nombre_programa_clean"]

In [7]:
# Modelo BERTa para embeddings
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [8]:
# Crear embeddings del entrenamiento (solo una vez)
X_train = df_train["texto_modelo"].tolist()
emb_train = model.encode(X_train, show_progress_bar=True, normalize_embeddings=True)

Batches:   0%|          | 0/161 [00:00<?, ?it/s]

In [9]:
# Clasificar por similitud (KNN por vecinos más cercanos)
nn = NearestNeighbors(n_neighbors=1, metric="cosine")
nn.fit(emb_train)

X_pred = df_pred["texto_modelo"].tolist()
emb_pred = model.encode(X_pred, show_progress_bar=True, normalize_embeddings=True)
dist, idx = nn.kneighbors(emb_pred)  
df_pred["match_score"] = 1 - dist.flatten() 
df_pred["match_index"] = idx.flatten()

Batches:   0%|          | 0/73 [00:00<?, ?it/s]

In [10]:
# Completar las columnas desde el match
cols_to_copy = ["codigo_grupo_1", "nombre_grupo_1", "codigo_grupo_3", "nombre_grupo_3"]
for c in cols_to_copy:
    df_pred[c] = df_train.iloc[df_pred["match_index"]][c].values

In [20]:
# Marcar para revisión manual
umbral = 0.75
df_pred["revision_manual"] = (df_pred["match_score"] < umbral).astype(int)
df_pred[df_pred["revision_manual"]==1][["nivel_academico","nombre_programa","match_score"] + cols_to_copy].head(10)

,nivel_academico,nombre_programa,match_score,codigo_grupo_1,nombre_grupo_1,codigo_grupo_3,nombre_grupo_3
5,Pregrado,Administración y Negocios Internacionales Moda...,0.743357,4,Ciencias Administrativas y Derecho,416,Negocios Internacionales
115,Pregrado,Idioma Inglés - PCPM,0.744268,2,Arte y Humanidades,231,Idiomas
263,Pregrado,Sin información,0.680904,2,Arte y Humanidades,222,Historia y Arqueología
310,Maestría,CALIDAD TOTAL,0.730031,7,"Ingeniería, Industria y Construcción",722,Industria y Productos Transformados (excepto a...
335,Maestría,DERECHI PRIVADO,0.685975,4,Ciencias Administrativas y Derecho,417,Administración Pública
347,Maestría,DERECHO PRIVADO,0.683819,4,Ciencias Administrativas y Derecho,422,Derecho Privado
370,Maestría,ENERGIA SOLAR,0.731318,7,"Ingeniería, Industria y Construcción",711,Electricidad y Energía
379,Maestría,EPIDEMOLOGIA,0.716223,9,Salud y bienestar,911,Odontología
380,Maestría,EPIODEMIOLOGIA,0.714007,9,Salud y bienestar,911,Odontología
397,Maestría,GANADERIA ANDINA,0.740814,2,Arte y Humanidades,222,Historia y Arqueología


In [12]:
# Guardar
salida = carpeta / "BD_programas_prediccion_clasificada.txt"
df_pred.to_csv(salida, sep="|", index=False, encoding="utf-8")